## Decision Tree

A decision tree is a popular machine learning algorithm used for both classification and regression tasks. It works by recursively splitting the data into subsets based on the feature that provides the best separation of the target variable. The resulting tree structure consists of nodes representing features, branches representing decisions, and leaves representing the final predictions.

In [33]:
import pandas as pd

# Define the dataset
data = {
    'Age': ['≤30', '≤30', '31…40', '>40', '>40', '>40', '31…40', '≤30', '≤30', '>40', '≤30', '31…40', '31…40', '>40'],
    'Income': ['high', 'high', 'high', 'medium', 'low', 'low', 'low', 'medium', 'low', 'medium',
               'medium', 'medium', 'high', 'medium'],
    'Student': ['no', 'no', 'no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no'],
    'Credit_Rating': ['Fair', 'Excellent', 'Fair', 'Fair', 'Fair', 'Excellent', 'Excellent', 'Fair',
                      'Fair', 'Fair', 'Excellent', 'Excellent', 'Fair', 'Excellent'],
    'Buy_XBOX': ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']
}

# Create DataFrame
df = pd.DataFrame(data)


# Split features and target BEFORE encoding
target = 'Buy_XBOX'
X = df.drop(target, axis=1)
y = df[target]

In [34]:
for i in range(len(df.columns)):
    unique_columns = df[df.columns[i]].unique()
    df[df.columns[i]] = df[df.columns[i]].map({unique_columns[value]: value for value in range(len(unique_columns))})

print(df)

    Age  Income  Student  Credit_Rating  Buy_XBOX
0     0       0        0              0         0
1     0       0        0              1         0
2     1       0        0              0         1
3     2       1        0              0         1
4     2       2        1              0         1
5     2       2        1              1         0
6     1       2        1              1         1
7     0       1        0              0         0
8     0       2        1              0         1
9     2       1        1              0         1
10    0       1        1              1         1
11    1       1        0              1         1
12    1       0        1              0         1
13    2       1        0              1         0


Entropy is a measure of impurity or disorder in a dataset. In the context of decision trees, it is used to determine how well a feature splits the data. The goal is to find the feature that minimizes the entropy after the split, which indicates a more homogeneous subset of data.

Formula:

$$Entropy(S) = -\sum_{i=1}^{n} p_i \log_2(p_i)$$

Where:
- $S$ is the set of data points.
- $n$ is the number of classes in the target variable.
- $p_i$ is the proportion of data points in class $i$.

First we will calculate the entropy of the target variable (Buy_XBOX) before splitting the data.

In [35]:
from math import log2

def entropy_target(target):
    p_list = []
    n = target.unique()  # Number of unique outcomes
    for i in range(len(n)):  
        # Calculate the p(class) of the outcome
        p = (target.value_counts().get(n[i], 0)) / (len(target))
        # print(f"P(Target = {target.name} -> Outcome {n[i]}): {p}")
        p_list.append(p) 
    # Apply the formula (add p > 0 to handle entropy = 0)
    entropy = sum(- p * log2(p) for p in p_list if p > 0)
    # print(f"E(Target = {target.name}): {entropy}")

    return entropy

e_target = entropy_target(y)
print(f"Entropy of the target variable: {e_target}")

Entropy of the target variable: 0.9402859586706311


Then we will calculate the entropy of each feature and determine which feature provides the best split based on the lowest entropy.

In [36]:
def entropy_feature(feature, target):
    df_combined = pd.concat([feature, target], axis=1)
    entropy_list = []
    n_feature = feature.unique()  # Number of unique classes
    n_target = target.unique()  # Number of unique outcomes
    # print("Feature =", feature.name)
    for i in range(len(n_feature)):
        p_list = []
        for j in range(len(n_target)):
            # Total number of samples for the class
            num_class = feature.value_counts().get(n_feature[i], 0)
            # Count the outcome for the current class, get the sub-dataset
            sub_df = df_combined[(df_combined[feature.name] == n_feature[i]) 
                                 & (df_combined[target.name] == n_target[j])]
            # Find the probability
            p = len(sub_df) / num_class if num_class != 0 else 0
            # print(f"+ P({n_feature[i]} -> Outcome = {n_target[j]}): {p}")
            p_list.append(p)
        # Compute the entropy
        entropy = sum(-p * log2(p) for p in p_list if p > 0)
        # print(f"- Entropy(Feature Class = {n_feature[i]}): {entropy}")
        # Append to list
        entropy_list.append(entropy)

    return entropy_list

# Calculate the entropy of each feature
for column in X.columns:
    e_feature = entropy_feature(X[column], y)
    print(f"Entropy of feature '{column}': {e_feature}")

Entropy of feature 'Age': [np.float64(0.9709505944546686), np.float64(0.0), np.float64(0.9709505944546686)]
Entropy of feature 'Income': [np.float64(1.0), np.float64(0.9182958340544896), np.float64(0.8112781244591328)]
Entropy of feature 'Student': [np.float64(0.9852281360342515), np.float64(0.5916727785823275)]
Entropy of feature 'Credit_Rating': [np.float64(0.8112781244591328), np.float64(1.0)]


Weighted Average is used to calculate the overall entropy after a split, taking into account the proportion of data points in each subset.

In [37]:
def weight_average(feature, target):
    # Combine feature and target into one dataframe
    df_combined = pd.concat([feature, target], axis=1)

    unique_values = feature.unique()
    total_samples = len(feature)

    weighted_entropy = 0  # Accumulator

    for value in unique_values:
        # Number of samples in this feature value
        num_class = feature.value_counts().get(value, 0)
        # Subset of data
        sub_df = df_combined[df_combined[feature.name] == value]
        # Entropy of the subset
        e_sub = entropy_target(sub_df[target.name])
        # Add weighted entropy
        weighted_entropy += (num_class / total_samples) * e_sub

    return weighted_entropy
    
# Calculate the overall weighted average entropy for each feature
for column in X.columns:
    e_feature = weight_average(X[column], y)
    print(f"Weighted Average of feature '{column}': {e_feature}")

Weighted Average of feature 'Age': 0.6935361388961918
Weighted Average of feature 'Income': 0.9110633930116763
Weighted Average of feature 'Student': 0.7884504573082896
Weighted Average of feature 'Credit_Rating': 0.8921589282623617


Information Gain is the difference between the entropy of the target variable before the split and the weighted average entropy after the split. The feature with the highest information gain is chosen for the split.

In [38]:
def information_gain(feature, target):
    e_target = entropy_target(target)
    e_feature = weight_average(feature, target)
    gain = e_target - e_feature
    print(f"Information Gain for feature '{feature.name}': {gain}")
    return gain

# Determine the best feature based on the highest information gain
best_feature = None
highest_gain = -1

for column in X.columns:
    gain = information_gain(X[column], y)
    if gain > highest_gain:
        highest_gain = gain
        best_feature = column

print(f"The best feature for splitting the data is: {best_feature}")

Information Gain for feature 'Age': 0.24674981977443933
Information Gain for feature 'Income': 0.02922256565895487
Information Gain for feature 'Student': 0.15183550136234159
Information Gain for feature 'Credit_Rating': 0.04812703040826949
The best feature for splitting the data is: Age


Recursive Splitting is the process of repeating the splitting process on each subset of data until a stopping criterion is met (e.g., maximum depth, minimum samples per leaf).

In [39]:
def recursive_splitting(x, y, depth=0):
    if len(y.unique()) == 1:
        return y.iloc[0]
    
    if depth == 0:
        print("Depth:", depth)
        print("Features:", x.columns)
        print("Target:", y.name)
    
    e_target = entropy_target(y)
    weight_average = []
    for i in range(len(x.columns)):
        unique_class = [x[x.columns[i]].value_counts().get(j, 0) for j in  x[x.columns[i]].unique()]
        entropy = entropy_feature(x[x.columns[i]], y)
        weight = sum((unique_class[j] / len(x)) * entropy[j] for j in range(len(unique_class)))
        weight_average.append(weight)

    information_gain = []
    for i in range(len(x.columns)):
        ig = e_target - weight_average[i]
        information_gain.append(ig)

    best_feature_index = information_gain.index(max(information_gain))
    best_feature = x.columns[best_feature_index]
    
    print(f"Best feature at depth {depth}: {best_feature}")
    
    tree = {best_feature: {}}
    
    for value in x[best_feature].unique():
        sub_x = x[x[best_feature] == value].drop(columns=best_feature)
        sub_y = y[x[best_feature] == value]
        
        tree[best_feature][value] = recursive_splitting(sub_x, sub_y, depth + 1)
    
    return tree

In [40]:
x = df.drop('Buy_XBOX', axis=1)
y = df['Buy_XBOX']

decision_tree = recursive_splitting(x, y)
print("Decision Tree:")
for key, value in decision_tree.items():
    print(f"{key}: {value}")

Depth: 0
Features: Index(['Age', 'Income', 'Student', 'Credit_Rating'], dtype='object')
Target: Buy_XBOX
Best feature at depth 0: Age
Best feature at depth 1: Student
Best feature at depth 1: Credit_Rating
Decision Tree:
Age: {np.int64(0): {'Student': {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}}, np.int64(1): np.int64(1), np.int64(2): {'Credit_Rating': {np.int64(0): np.int64(1), np.int64(1): np.int64(0)}}}
